# Turkish Legal RAG - Embedding Fine-Tuning

Bu notebook embedding tuning ablation deneyi içindir.

Amaç:

```text
Base dense retrieval: BAAI/bge-m3
Tuned dense retrieval: BAAI/bge-m3 + legal triplet fine-tuning
```

Eğitim datası:

- `clean_train.jsonl`
- `clean_dev.jsonl`

Bu dosyalarda pozitifler kesin `positive_parent_id` üzerinden, negatifler ise BM25 hard negative olarak hazırlanmıştır.


In [ ]:
!nvidia-smi
!echo "Input files:"
!find /kaggle/input -maxdepth 6 -type f | sort | head -300


## 1. Paket Kurulumu


In [ ]:
!pip install -q -U sentence-transformers faiss-cpu datasets tqdm


## 2. Dosyaları Working Klasörüne Kopyala


In [ ]:
from pathlib import Path
import shutil

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/legal-rag")

for path in [
    WORK_DIR / "scripts",
    WORK_DIR / "data/processed",
    WORK_DIR / "data/index",
    WORK_DIR / "data/reranker",
    WORK_DIR / "data/eval",
    WORK_DIR / "models",
]:
    path.mkdir(parents=True, exist_ok=True)

def find_input_file(name: str) -> Path:
    matches = sorted(INPUT_ROOT.rglob(name))
    if not matches:
        raise FileNotFoundError(f"{name} not found under {INPUT_ROOT}. Kaggle dataset'e ekledin mi?")
    return matches[0]

def copy_required(name: str, dest_dir: Path) -> Path:
    src = find_input_file(name)
    dst = dest_dir / name
    shutil.copy2(src, dst)
    print(f"Copied {src} -> {dst}")
    return dst

for name in [
    "train_embedding_model.py",
    "build_faiss_index.py",
    "evaluate_retrieval.py",
    "search_faiss.py",
]:
    copy_required(name, WORK_DIR / "scripts")

for name in ["retrieval_chunks.json", "retrieval_corpus.json"]:
    copy_required(name, WORK_DIR / "data/processed")

for name in ["clean_train.jsonl", "clean_dev.jsonl"]:
    copy_required(name, WORK_DIR / "data/reranker")

copy_required("qa_benchmark_gold.csv", WORK_DIR / "data/eval")

# Baseline BGE-M3 index varsa kopyalıyoruz; yoksa sadece tuned index kurulur.
for name in ["faiss_bge_m3.index", "metadata_bge_m3.json", "index_config_bge_m3.json"]:
    try:
        copy_required(name, WORK_DIR / "data/index")
    except FileNotFoundError:
        print(f"Baseline index file missing, skipping: {name}")

print("\nWorking files:")
!find /kaggle/working/legal-rag -maxdepth 4 -type f | sort


## 3. Training Data Kontrolü


In [ ]:
import json
from collections import Counter
from pathlib import Path

def load_jsonl(path):
    return [json.loads(line) for line in Path(path).open(encoding="utf-8") if line.strip()]

train_rows = load_jsonl(WORK_DIR / "data/reranker/clean_train.jsonl")
dev_rows = load_jsonl(WORK_DIR / "data/reranker/clean_dev.jsonl")

print("train rows:", len(train_rows), Counter(row["label"] for row in train_rows))
print("dev rows:", len(dev_rows), Counter(row["label"] for row in dev_rows))
print("train query groups:", len({row["query_id"] for row in train_rows}))
print("dev query groups:", len({row["query_id"] for row in dev_rows}))


## 4. Embedding Model Fine-Tuning

BGE-M3 büyük model olduğu için ilk denemede güvenli ayarlar:

```text
epochs = 1
batch_size = 1
max_triplets_per_query = 2
loss = TripletLoss
```

Eğer Kaggle GPU rahat çalışırsa sonradan `batch-size 2` veya `max-triplets-per-query 3` denenebilir.


In [ ]:
!rm -rf /kaggle/working/legal-rag/models/legal-bge-m3-embedding

!CUDA_VISIBLE_DEVICES=0 python /kaggle/working/legal-rag/scripts/train_embedding_model.py   --train /kaggle/working/legal-rag/data/reranker/clean_train.jsonl   --dev /kaggle/working/legal-rag/data/reranker/clean_dev.jsonl   --base-model BAAI/bge-m3   --output-dir /kaggle/working/legal-rag/models/legal-bge-m3-embedding   --epochs 1   --batch-size 1   --learning-rate 1e-5   --triplet-margin 0.25   --max-triplets-per-query 2   --max-seq-length 512   --device cuda   --evaluation-steps 200   --save-best-model


## 5. Fine-Tuned Embedding Model Kontrolü


In [ ]:
!ls -lh /kaggle/working/legal-rag/models/legal-bge-m3-embedding
!test -f /kaggle/working/legal-rag/models/legal-bge-m3-embedding/modules.json
!cat /kaggle/working/legal-rag/models/legal-bge-m3-embedding/training_config.json


## 6. Fine-Tuned Embedding ile FAISS Index Kur


In [ ]:
!mkdir -p /kaggle/working/legal-rag/data/index_tuned_embedding

!python /kaggle/working/legal-rag/scripts/build_faiss_index.py   --chunks /kaggle/working/legal-rag/data/processed/retrieval_chunks.json   --index-out /kaggle/working/legal-rag/data/index_tuned_embedding/faiss_legal_bge_m3.index   --metadata-out /kaggle/working/legal-rag/data/index_tuned_embedding/metadata_legal_bge_m3.json   --config-out /kaggle/working/legal-rag/data/index_tuned_embedding/index_config_legal_bge_m3.json   --model /kaggle/working/legal-rag/models/legal-bge-m3-embedding   --device cuda   --batch-size 8


## 7. Gold Benchmark Evaluation - Tuned Dense


In [ ]:
!python /kaggle/working/legal-rag/scripts/evaluate_retrieval.py   --benchmark /kaggle/working/legal-rag/data/eval/qa_benchmark_gold.csv   --corpus /kaggle/working/legal-rag/data/processed/retrieval_corpus.json   --chunks /kaggle/working/legal-rag/data/processed/retrieval_chunks.json   --index /kaggle/working/legal-rag/data/index_tuned_embedding/faiss_legal_bge_m3.index   --metadata /kaggle/working/legal-rag/data/index_tuned_embedding/metadata_legal_bge_m3.json   --config /kaggle/working/legal-rag/data/index_tuned_embedding/index_config_legal_bge_m3.json   --mode dense   --embedding-device cuda   --top-k 10   --output /kaggle/working/legal-rag/data/eval/eval_dense_tuned_embedding.json


## 8. Gold Benchmark Evaluation - Tuned Dense + BM25 Hybrid


In [ ]:
!python /kaggle/working/legal-rag/scripts/evaluate_retrieval.py   --benchmark /kaggle/working/legal-rag/data/eval/qa_benchmark_gold.csv   --corpus /kaggle/working/legal-rag/data/processed/retrieval_corpus.json   --chunks /kaggle/working/legal-rag/data/processed/retrieval_chunks.json   --index /kaggle/working/legal-rag/data/index_tuned_embedding/faiss_legal_bge_m3.index   --metadata /kaggle/working/legal-rag/data/index_tuned_embedding/metadata_legal_bge_m3.json   --config /kaggle/working/legal-rag/data/index_tuned_embedding/index_config_legal_bge_m3.json   --mode hybrid   --embedding-device cuda   --top-k 10   --output /kaggle/working/legal-rag/data/eval/eval_hybrid_tuned_embedding.json


## 9. Sonuçları Göster


In [ ]:
import json
from pathlib import Path

for name in [
    "eval_dense_tuned_embedding.json",
    "eval_hybrid_tuned_embedding.json",
]:
    path = WORK_DIR / "data/eval" / name
    data = json.loads(path.read_text(encoding="utf-8"))
    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)
    print(json.dumps(data["summary"], ensure_ascii=False, indent=2))


## 10. Output Olarak Saklanacaklar

Kaggle notebook bitince şunları saklayın:

- `/kaggle/working/legal-rag/models/legal-bge-m3-embedding`
- `/kaggle/working/legal-rag/data/index_tuned_embedding`
- `/kaggle/working/legal-rag/data/eval/eval_dense_tuned_embedding.json`
- `/kaggle/working/legal-rag/data/eval/eval_hybrid_tuned_embedding.json`

Rapor için karşılaştırma:

```text
Dense BGE-M3 baseline vs Fine-tuned dense embedding
Hybrid BGE-M3 baseline vs Fine-tuned dense + BM25
```
